# פרק 7: בניית אפליקציות צ'אט
## התחלה מהירה עם OpenAI API

מחברת זו מותאמת מ-[Azure OpenAI Samples Repository](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) שכוללת מחברות שמקשרות לשירותי [Azure OpenAI](notebook-azure-openai.ipynb).

ממשק ה-Python של OpenAI API עובד גם עם דגמי Azure OpenAI, עם כמה התאמות. למידע נוסף אודות ההבדלים כאן: [כיצד לעבור בין נקודות הקצה של OpenAI ו-Azure OpenAI באמצעות Python](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/switching-endpoints?WT.mc_id=academic-109527-jasmineg)


# סקירה כללית  
"מודלים לשוניים גדולים הם פונקציות שממפות טקסט לטקסט. בהינתן מחרוזת טקסט כנכנס, מודל לשוני גדול מנסה לנבא את הטקסט שיבוא אחרי זה"(1). מחברת ה"מתחילים המהיר" הזו תציג למשתמשים מושגים ברמה גבוהה של LLM, דרישות חבילה בסיסיות להתחלה עם AML, הקדמה רכה לתכנון פקודות, וכמה דוגמאות קצרות לשימושים שונים. 


## תוכן העניינים  

[סקירה כללית](#overview)  
[כיצד להשתמש בשירות OpenAI](#how-to-use-openai-service)  
[1. יצירת שירות OpenAI שלך](#1.-creating-your-openai-service)  
[2. התקנה](#2.-installation)    
[3. אישורי גישה](#3.-credentials)  

[מקרי שימוש](#use-cases)    
[1. סיכום טקסט](#1.-summarize-text)  
[2. סיווג טקסט](#2.-classify-text)  
[3. יצירת שמות חדשים למוצרים](#3.-generate-new-product-names)  
[4. כוונון מעודן של מסווג](#4.fine-tune-a-classifier)  

[מקורות](#references)


### בנה את הפקודה הראשונה שלך  
תרגיל קצר זה ייתן מבוא בסיסי להגשת פקודות למודל OpenAI עבור משימה פשוטה של "סיכום".


**שלבים**:  
1. התקן את ספריית OpenAI בסביבת הפייתון שלך  
2. טען ספריות עזר סטנדרטיות והגדר את האישורים האבטחתיים הרגילים שלך לשירות OpenAI שיצרת  
3. בחר מודל למשימה שלך  
4. צור פקודה פשוטה עבור המודל  
5. הגש את הבקשה שלך ל-API של המודל!  


### 1. התקנת OpenAI


In [ ]:
%pip install openai python-dotenv

### 2. ייבא ספריות עזר ויצר אישורים


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY","")
assert API_KEY, "ERROR: OpenAI Key is missing"

client = OpenAI(
    api_key=API_KEY
    )


### 3. מציאת הדגם המתאים  
דגמים כמו GPT-4o ו-GPT-4o mini יכולים להבין וליצור שפה טבעית, וניתנים לשימוש בפלטפורמת OpenAI ברמות שונות של עוצמה ומהירות המתאימות למשימות שונות.


In [ ]:
# Select a general purpose chat model
model = "gpt-5-mini"


## 4. עיצוב פרומפט  

"הקסם של מודלים שפתיים גדולים הוא שבאמצעות אימון למזער את שגיאת התחזית הזו על כמות עצומה של טקסט, המודלים לומדים בסופו של דבר מושגים שימושיים עבור תחזיות אלו. לדוגמה, הם לומדים מושגים כמו"(1):

* איך לאיית
* איך עובדת הדקדוק
* איך לפרפרז
* איך לענות על שאלות
* איך לנהל שיחה
* איך לכתוב בשפות רבות
* איך לקודד
* וכן הלאה.

#### איך לשלוט במודל שפה גדול  
"מבין כל הקלטים למודל שפה גדול, הפומפט הוא בהחלט המשפיע ביותר(1).

ניתן לגרום למודלים שפתיים גדולים לייצר פלט בכמה דרכים:

הוראה: אמור למודל מה אתה רוצה
השלמה: גרום למודל להשלים את תחילת מה שאתה רוצה
הדגמה: הראה למודל מה אתה רוצה, באמצעות אחד מהבאים:
מספר דוגמאות בפרומפט
מאות או אלפי דוגמאות במסד נתוני אימון לכיול עדין"



#### יש שלושה קווים מנחים בסיסיים ליצירת פרומפטים:

**הראה ותגיד**. היה ברור למה אתה רוצה בין אם באמצעות הוראות, דוגמאות, או שילוב של השניים. אם אתה רוצה שהמודל ימיין רשימת פריטים לפי סדר אלפביתי או יסווג פסקה לפי סנטימנט, הראה לו שזה מה שאתה רוצה.

**ספק נתונים איכותיים**. אם אתה מנסה לבנות מסווג או לגרום למודל לעקוב אחרי תבנית, וודא שיש מספיק דוגמאות. הקפד לבדוק את הדוגמאות שלך — המודל בדרך כלל חכם מספיק כדי לראות טעויות איות בסיסיות ולספק תגובה, אבל ייתכן שגם יחשוב שזה בכוונה וזה יכול להשפיע על התגובה.

**בדוק את ההגדרות שלך.** ההגדרות temperature ו-top_p שולטות עד כמה המודל דטרמיניסטי ביצירת תגובה. אם אתה מבקש תגובה שיש לה תשובה אחת נכונה בלבד, תרצה להוריד אותן. אם אתה מחפש תגובות מגוונות יותר, תעדיף להעלות אותן. הטעות מספר אחת שאנשים עושים עם ההגדרות האלה היא להניח שהן שולטות ב"חוכמה" או "יצירתיות".


מקור: https://learn.microsoft.com/azure/ai-foundry/openai/overview


### 5. הגש!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


### חזור על אותה קריאה, איך התוצאות להשוות? 


In [ ]:

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


## סכם טקסט  
#### אתגר  
סכם טקסט על ידי הוספת 'tl;dr:' בסוף קטע טקסט. שים לב כיצד המודל מבין כיצד לבצע מספר משימות ללא הוראות נוספות. אתה יכול להתנסות עם פקודות תיאוריות יותר מ-tl;dr כדי לשנות את התנהגות המודל ולהתאים את הסיכום שתקבל(3).  

עבודה אחרונה הראתה שיפורים משמעותיים בהרבה משימות NLP ובמדדי ביצועים על ידי אימון מקדים על מאגר טקסט גדול ולאחר מכן כוונון עדין למשימה ספציפית. בעוד שבדרך כלל הארכיטקטורה אינה תלויה במשימה, שיטה זו עדיין דורשת מערכי נתונים מותאמים למשימה עם אלפי או עשרות אלפי דוגמאות. לעומת זאת, בני אדם יכולים בדרך כלל לבצע משימת שפה חדשה רק ממספר דוגמאות מועט או מהוראות פשוטות - משהו שמערכות NLP הנוכחיות עדיין מתקשות מאוד לעשות. כאן אנו מראים כי הגדלת גודל מודלי השפה משפרת במידה רבה את הביצועים ללא תלות במשימה, במספר דוגמאות מועטות, ולעיתים אפילו מגיעה לתחרותיות עם גישות הכוונון העדין המתקדמות ביותר הקודמות.  



Tl;dr


# תרגילים למספר מקרים שימוש  
1. סיכום טקסט  
2. סיווג טקסט  
3. יצירת שמות מוצרים חדשים


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## לסווג טקסט  
#### אתגר  
לסווג פריטים לקטגוריות המסופקות בזמן האינפרנס. בדוגמה הבאה, אנו מספקים גם את הקטגוריות וגם את הטקסט לסיווג בהנחיה (*playground_reference). 

פנייה מלקוח: שלום, אחד המקשים במקלדת הלפטופ שלי נשרט לאחרונה ואצטרך להחליף אותו:

קטגוריה מסווגת:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## צור שמות חדשים למוצרים  
#### אתגר  
צור שמות למוצרים מתוך מילים לדוגמה. כאן כללנו בפרומפט מידע על המוצר שאנו הולכים ליצור עבורו שמות. בנוסף, סיפקנו דוגמה דומה כדי להראות את התבנית שאנו רוצים לקבל. גם קבענו את ערך הטמפרטורה גבוה כדי להגדיל את הרנדומליות ואת התגובות החדשניות יותר.  

תיאור המוצר: מכשיר להכנת מילקשייק בבית  
מילים בסיסיות: מהיר, בריא, קומפקטי.  
שמות המוצר: HomeShaker, Fit Shaker, QuickShake, Shake Maker  

תיאור המוצר: זוג נעליים שיכול להתאים לכל מידה של רגל.  
מילים בסיסיות: מתאים, אדפטיבי, מתאים-לכל.  


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}],
  store=False,)

response.output_text


# מקורות  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [Best practices for fine-tuning GPT models to classify text](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# לקבלת עזרה נוספת  
[OpenAI Commercialization Team](AzureOpenAITeam@microsoft.com) 


# תורמים
* Louis Li  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**כתב ויתור**:
מסמך זה תורגם באמצעות שירות תרגום אוטומטי [Co-op Translator](https://github.com/Azure/co-op-translator). למרות שאנו שואפים לדיוק, יש לקחת בחשבון שתרגומים אוטומטיים עלולים להכיל שגיאות או אי-דיוקים. יש להחשיב את המסמך המקורי בשפתו הטבעית כמקור הסמכות. למידע קריטי מומלץ להשתמש בתרגום מקצועי על ידי מתרגם אדם. אנו לא אחראים לכל אי-הבנה או פירוש שגוי הנובע מהשימוש בתרגום זה.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
